In [2]:
from pathlib import Path
import pandas as pd
import nibabel as nib
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score
import numpy as np
import pickle
import random
import SimpleITK as sitk
import cv2
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
import numpy as np
from torchvision import transforms

In [3]:
res_size = (96, 48, 128)

base_folder = Path(r"/home/varhai/PRIMUS/data/83_IMPC_DATA/Bioimaging/SCREENad.Morpho&BodyComp")
skull_file = base_folder / "combined_skull_analysis.xlsx"
skull_df = pd.read_excel(skull_file)

# Filter subjects with skull data which have sutures

In [4]:
skull_df["sutures_status"] = skull_df['Comment'].apply(lambda x: ("sutures"in str(x).lower() and "open" in str(x).lower()
                                                                 # and "severe" in str(x).lower()
                                                                 ))
skull_df = skull_df[skull_df['proc_dist']<3000]

In [5]:
# get 2 classes for sutures (equal number of sutures and no sutures)
sutures_df = skull_df[skull_df['sutures_status']==True]
no_sutures_df = skull_df[skull_df['sutures_status']==False]
no_sutures_df = no_sutures_df.sample(n=sutures_df.shape[0], random_state=42)

In [6]:
# generate train and test split
train_df = pd.concat([sutures_df.iloc[:int(sutures_df.shape[0]*0.8)], no_sutures_df.iloc[:int(no_sutures_df.shape[0]*0.8)]])
test_df = pd.concat([sutures_df.iloc[int(sutures_df.shape[0]*0.8):], no_sutures_df.iloc[int(no_sutures_df.shape[0]*0.8):]])
# randomize the dataset
train_df = train_df.sample(frac=1, random_state=42)
test_df = test_df.sample(frac=1, random_state=42)

In [7]:
# resize the images to 128x128x128
def resize_image(img, size=(128, 128, 128)):
    """
    Resize a 3D image to the given size.

    Parameters:
    img (numpy.ndarray): The input 3D image.
    size (tuple): The target size for each axis (depth, height, width).

    Returns:
    numpy.ndarray: The resized 3D image.
    """
    img = torch.tensor(img, dtype=torch.float32)  # Convert to a tensor
    img = img.unsqueeze(0).unsqueeze(0)  # Add batch and channel dimensions
    img = F.interpolate(img, size=size, mode='trilinear', align_corners=False)  # Resize
    img = img.squeeze(0).squeeze(0)  # Remove batch and channel dimensions
    return img.numpy()


def resize_data(data, size=128):
    for i in range(len(data)):
        data[i]['image'] = resize_image(data[i]['image'], size)
    return data

def read_nifty_image(file_path):
    img = nib.load(file_path)
    return img.get_fdata()

def find_location_subject(cohort_id,subject,folder_par: Path):
    ar = list(folder_par.glob(f"*{cohort_id}*/skull_shape_workdir/{subject}"))
    subj = None
    if len(ar)>0:
        subj = ar[0]
    return subj

def find_3d_bounding_box(mask):
    # Get the indices of the non-zero elements
    non_zero_indices = np.argwhere(mask)
    
    # Get the minimum and maximum indices along each axis
    z_min, y_min, x_min = non_zero_indices.min(axis=0)
    z_max, y_max, x_max = non_zero_indices.max(axis=0)
    
    # Return the bounding box coordinates
    return (z_min, y_min, x_min), (z_max, y_max, x_max)


def read_nifty_subject(folder: Path):
    
    mask = read_nifty_image(folder / "mask_skull_top.nii.gz")
    img = read_nifty_image(folder / "coregistered_head.nii.gz")
    
    return (img).astype(np.float32) ,mask 
    

In [8]:
import os
def read_subjs(df, process = lambda x : x, is_process= False, folder = None):
    results = []
    for i in tqdm(range(df.shape[0])):
        subj = dict(df.iloc[i])
        s_id = subj['Unnamed: 0']
        coh = subj['COHORT_ID']
        clas_res = subj['sutures_status']
        
        loc_dir = find_location_subject(coh,s_id,base_folder)
        if loc_dir:
            img,mask = read_nifty_subject(loc_dir)
            ids = find_3d_bounding_box(mask)
            #img = img * mask
            if is_process:
                mask = morphology.binary_dilation(mask)
                for _ in range(4):
                    mask = morphology.binary_dilation(mask)
                img[img<34]= 34
                img[img>50] = 50
                img = process(img*mask)
                
                nib.save(nib.Nifti1Image(img, np.eye(4)), os.path.join(folder, f"{i}_{clas_res}.nii.gz"))
            print(np.array(ids[1]) - np.array(ids[0]))
            #img = img[ids[0][0]:ids[1][0],ids[0][1]:ids[1][1],ids[0][2]:ids[1][2]]
            #img = resize_image(img, size=res_size)
            res = { "class": clas_res,  "image": img}
            results.append(res)
    return results
train_data = read_subjs(train_df,
                        #process_image,
                        is_process=True, folder=r'/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/workdir/test_data_sutures')
with open(r'/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/workdir/train_data_sutures.pkl', 'wb') as f:
    pickle.dump(train_data, f)
test_data = read_subjs(test_df)
# pickle the data


with open(r'/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/workdir/test_data_sutures.pkl', 'wb') as f:
    pickle.dump(test_data, f)
    


  0%|          | 0/984 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [9]:
# load the data
with open(r'/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/workdir/train_data_sutures.pkl', 'rb') as f:
    train_data = pickle.load(f)
with open(r'/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/workdir/test_data_sutures.pkl', 'rb') as f:
    test_data = pickle.load(f)

In [10]:
train_data = resize_data(train_data,res_size)
test_data = resize_data(test_data,res_size)


In [60]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv3d(in_channels, out_channels, kernel_size, stride, padding)
        self.bn1 = nn.BatchNorm3d(out_channels)
        self.conv2 = nn.Conv3d(out_channels, out_channels, kernel_size, 1, padding)
        self.bn2 = nn.BatchNorm3d(out_channels)
        
        if in_channels != out_channels:
            self.adjust = nn.Conv3d(in_channels, out_channels, kernel_size=1, stride=stride)
        else:
            self.adjust = None

    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.adjust is not None:
            residual = self.adjust(residual)
        out += residual
        return F.relu(out)

class Net(nn.Module):
    def __init__(self, input_shape):
        super(Net, self).__init__()
        self.layer1 = ResidualBlock(1, 32)
        self.layer2 = ResidualBlock(32, 64)
        self.layer3 = ResidualBlock(64, 128)
        self.layer4 = ResidualBlock(128, 256)
        
        self.pool = nn.MaxPool3d(2, 2)
        self.dropout = nn.Dropout3d(0.3)
        
        self.feature_size = self._get_conv_output(input_shape)
        
        self.fc1 = nn.Linear(self.feature_size, 512)
        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, 1)

    def _get_conv_output(self, shape):
        with torch.no_grad():
            x = torch.zeros(1, *shape)
            x = self.layer1(x)
            x = self.pool(x)
            x = self.layer2(x)
            x = self.pool(x)
            x = self.layer3(x)
            x = self.pool(x)
            x = self.layer4(x)
            x = self.pool(x)
            return x.numel()

    def forward(self, x):
        x = self.layer1(x)
        x = self.pool(x)
        x = self.layer2(x)
        x = self.pool(x)
        x = self.layer3(x)
        x = self.pool(x)
        x = self.layer4(x)
        x = self.pool(x)
        
        x = x.view(x.size(0), -1)  # Flatten the tensor
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = torch.sigmoid(self.fc3(x))
        
        return x

In [87]:



class MouseSuturesDataset(Dataset):
    def __init__(self, data, augment=False, converter = None):
        self.data = data
        self.augment = augment
        self.images_processed = {}
        self.converter = converter

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image = self.data[idx]['image']
        min_im = image.min()
        max_im = image.max()
        image = (image - min_im) / (max_im - min_im)
        image = torch.tensor(image, dtype=torch.float32)

        image = image.unsqueeze(0)  # Add channel dimension
        if random.random()>0.5:
            if self.augment:
                image = self._augment_image(image)
                image = converter(image)
        else:
            if idx in self.images_processed:
                image = self.images_processed[idx]
            else:
                self.images_processed[idx] = converter(image)
                

        label = self.data[idx]['class']
        label = self._convert_class_to_int(label)
        return image, label

    def _convert_class_to_int(self, class_value):
        return torch.tensor(int(class_value), dtype=torch.long)

    def _augment_image(self, image):
        # Randomly choose which augmentation to apply
        if random.random() > 0.2:
            image = self._random_shift(image)
        if random.random() > 0.2:
            image = self._add_gaussian_noise(image)
        return image

    def _random_shift(self, image, max_shift=5):
        # Randomly shift the image along each axis
        shift_d, shift_h, shift_w = np.random.randint(-max_shift, max_shift+1, size=3)
        image = torch.roll(image, shifts=(shift_d, shift_h, shift_w), dims=(1, 2, 3))
        return image

    def _add_gaussian_noise(self, image, mean=0, std=0.1):
        noise = torch.randn(image.size()) * std + mean
        image = image + noise
        image = torch.clamp(image, 0, 1)  # Ensure values are within [0, 1]
        return image

# Example usage:


In [63]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Transform to convert single-channel image to three-channel image
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor()
])
# Extract features using modified ResNet
def extract_features(model, images_2d):
    model.eval()
    features = []
    with torch.no_grad():
        for image_2d in images_2d:
            image_2d = transform(image_2d).unsqueeze(0).float().to(device)  # Add batch and channel dimensions
            feature = model(image_2d)
            feature = torch.flatten(feature, 1)  # Flatten the output
            features.append(feature)
    return torch.stack(features)


best_val_loss = 1000

vgg = models.vgg16(pretrained=True).to(device)
print(vgg.classifier)
#for param in vgg.parameters():
#    param.requires_grad = False
    
# Replace the classifier part with a custom one
vgg.classifier[6] = nn.Linear(4096, 1).to(device)

num_epochs = 500
transform = transforms.Compose([
    transforms.ToPILImage(),
#    transforms.Resize((128, 128)),#transforms.Resize((224, 224)),  # Resize images to fit VGG16 input
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor()
    ,transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalize according to VGG16 training data
])
train_dataset = MouseSuturesDataset(train_data, True)
test_dataset = MouseSuturesDataset(test_data, False)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=1, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False)

criterion = nn.L1Loss()
optimizer = torch.optim.Adam(vgg.classifier[6].parameters(), lr=1e-5)

early_stopping_patience = 20

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.92)
epoch_losses = []
val_losses = []
for epoch in range(500):
    vgg.train()
    running_loss = 0.0
    for inputs, labels in tqdm(train_loader):
        
        inputs, labels = transform(inputs.squeeze(0).float()).to(device), labels.float().to(device)
        
        optimizer.zero_grad()
        outputs = vgg(inputs.unsqueeze(0))
        #print(outputs)
        #print(outputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    epoch_loss = running_loss / len(train_loader.dataset)
    vgg.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = transform(inputs.squeeze(0).float()).to(device), labels.float().to(device)
            outputs = vgg(inputs.unsqueeze(0))
            loss = criterion(outputs, labels)
            val_loss += loss.item() #* inputs.size(0)
    
    val_loss = val_loss / len(test_loader.dataset)
    val_losses.append(val_loss)
    scheduler.step()
    
    print(f'Epoch {epoch+1}/{num_epochs}, Training Loss: {epoch_loss:.4f}, Validation Loss: {val_loss:.4f}')
    epoch_losses.append(epoch_loss)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(vgg.state_dict(), 'best_model.pth')
    else:
        patience_counter += 1
        if patience_counter >= early_stopping_patience:
            print("Early stopping triggered")
            break  

  0%|          | 0/984 [00:00<?, ?it/s]

Epoch 1/50, Training Loss: 0.5010, Validation Loss: 0.5004


  0%|          | 0/984 [00:00<?, ?it/s]

Epoch 2/50, Training Loss: 0.5110, Validation Loss: 0.5000


  0%|          | 0/984 [00:00<?, ?it/s]

Epoch 3/50, Training Loss: 0.5084, Validation Loss: 0.5000


  0%|          | 0/984 [00:00<?, ?it/s]

Epoch 4/50, Training Loss: 0.5079, Validation Loss: 0.5000


  0%|          | 0/984 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [37]:
for i in range(len(test_dataset)):
    if test_dataset[i][1] == 0:
        a = model(test_dataset[i][0])
        # print index of the class with the highest probability
        
        print(a)

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument mat2 in method wrapper_mm)

In [ ]:
import numpy as np
from skimage import exposure, filters, morphology
import matplotlib.pyplot as plt
from skimage.io import imread, imsave

def enhance_contrast(image):
    # Enhance contrast using histogram equalization
    return exposure.equalize_hist(image)

def segment_bones(image):
    # Thresholding to segment bones
    threshold_value = filters.sobel(image)
    #binary_image = image > threshold_value
    return threshold_value

def enhance_edges(binary_image):
    # Enhance edges using morphological operations
    edges = morphology.dilation(binary_image, morphology.ball(3))
    return edges

def process_image(image):
    # Load the 3D image
    x = image
    # Enhance contrast
    #x = enhance_contrast(image)

    # Segment bones
    x = segment_bones(x)

    # Enhance edges
    
    #enhanced_edges = enhance_edges(bones_segmented)
    
    return x

In [26]:
# save copies of nifty images from dataset into folder (create folder if not exist)
import os
import shutil



def save_nifty_images(data, folder, func=(lambda x : x) ):
    if not os.path.exists(folder):
        os.makedirs(folder)
    for i in range(len(data)):
        img = data[i]['image']
        img = img.astype(np.float32)
        img = func(img)
        img = img.astype(np.float32)
        img = nib.Nifti1Image(img, np.eye(4))
        #save class in name of the file (0 - no sutures, 1 - sutures)
        nib.save(img, os.path.join(folder, f"{i}_{data[i]['class']}.nii.gz"))
    
save_nifty_images(test_data, r'/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/workdir/test_data_sutures'#,process_image
                 )

In [ ]:
import skimage

In [ ]:
test_df.to_csv("/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/workdir/test_data_sutures/test_names.csv")

In [ ]:
sutures_df.describe()

In [ ]:
import torchvision

In [ ]:
torchvision.models.googlenet()

In [102]:


# Function to convert 3D image to 2D slices
def convert_3d_to_2d_slices(image_3d):
    slices = []
    for i in range(image_3d.shape[2]):
        slice_2d = image_3d[:, :, i]
        slices.append(slice_2d)
    return slices

# Transform to convert single-channel image to three-channel image
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor()
])
# Extract features using modified ResNet
def extract_features(model, images_3d):
    model.eval()
    features = []
    with torch.no_grad():
        for image_3d in images_3d:
            slices = convert_3d_to_2d_slices(image_3d)
            slice_features = []
            for slice_2d in slices:
                slice_2d = transform(slice_2d).unsqueeze(0).float().to(device)  # Add batch and channel dimensions
                feature = model(slice_2d)
                #print(feature.size())
                feature = torch.flatten(feature, 1)  # Flatten the output
                slice_features.append(feature)
            features.append(torch.stack(slice_features))  # Aggregate slice features
    return torch.stack(features)

best_val_loss = 1000

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(in_features=1000, out_features=500)
        self.fc2 = nn.Linear(in_features=500, out_features=50)
        self.fc3 = nn.Linear(in_features=50, out_features=1)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        #x = torch.flatten(x)
        
        return x.float()
early_stopping_patience = 20
train_dataset = MouseSuturesDataset(train_data, augment=True)
test_dataset = MouseSuturesDataset(test_data, augment=False)

#model = Net().to(device)#nn.Linear(in_features=1000, out_features=2).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.000001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.92)
criterion = nn.L1Loss()
epoch_losses = []
val_losses = []
for epoch in range(500):
    model.train()
    running_loss = 0.0
    for inputs, labels in tqdm(train_loader):
        inputs, labels = extract_features(resnet,inputs).float(), labels.float().to(device)
        
        #print(inputs.size())
        optimizer.zero_grad()
        outputs = model(inputs)
        #print(outputs)
        loss = criterion(outputs.unsqueeze(0).mean(), labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
    
    epoch_loss = running_loss / len(train_loader.dataset)
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = extract_features(resnet,inputs), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs.unsqueeze(0).mean(), labels)
            val_loss += loss.item() * inputs.size(0)
    
    val_loss = val_loss / len(test_loader.dataset)
    val_losses.append(val_loss)
    scheduler.step()
    
    print(f'Epoch {epoch+1}/{num_epochs}, Training Loss: {epoch_loss:.4f}, Validation Loss: {val_loss:.4f}')
    epoch_losses.append(epoch_loss)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        patience_counter += 1
        if patience_counter >= early_stopping_patience:
            print("Early stopping triggered")
            break

    

  0%|          | 0/984 [00:00<?, ?it/s]

Epoch 1/50, Training Loss: 0.4292, Validation Loss: 0.4197


  0%|          | 0/984 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [103]:
true_positives = 0
true_negatives = 0
false_positives = 0
false_negatives = 0
# load the best model
model.load_state_dict(torch.load('best_model.pth'))
for inputs, labels in tqdm(test_loader):
    inputs, labels = extract_features(resnet,inputs), labels.to(device)
    outputs = model(inputs)
    predicted_class = outputs.mean() > 0.5
    true_class = labels
    if true_class == 1 and predicted_class == 1:
        true_positives += 1
    elif true_class == 0 and predicted_class == 0:
        true_negatives += 1
    elif true_class == 0 and predicted_class == 1:
        false_positives += 1
    elif true_class == 1 and predicted_class == 0:
        false_negatives += 1

accuracy = (true_positives + true_negatives) / (true_positives + true_negatives + false_positives + false_negatives)
precision = true_positives / (true_positives + false_positives)
recall = true_positives / (true_positives + false_negatives)
f1_score = 2 * precision * recall / (precision + recall)
print(f'Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1_score:.4f}')
        #print(outputs.argmax(0))

  0%|          | 0/246 [00:00<?, ?it/s]

Accuracy: 0.6789, Precision: 0.6310, Recall: 0.8618, F1 Score: 0.7285


In [94]:
print(val_loss,epoch_loss,epoch)

86.4023798420094 0.4342041813065222 139


In [95]:
best_val_loss

0.4179309828555196